# `prompts.py` test

`prompts.py` only builds strings -- no model or database calls -- so most of this notebook is pure unit testing (free, deterministic). The last section is a real integration test: build a prompt with the actual schema, send it to the configured model via `llm.py`, and execute the result against `data/olist.db` (small real API cost, same as `04_llm_pipeline.ipynb`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import prompts
from config import settings

## Static system prompts

In [2]:
# SQL_SYSTEM must be assembled from the three rule blocks, in order
assert prompts.DIALECT_RULES in prompts.SQL_SYSTEM
assert prompts.SCHEMA_RULES in prompts.SQL_SYSTEM
assert prompts.OUTPUT_RULES in prompts.SQL_SYSTEM
assert prompts.SQL_SYSTEM.index(prompts.DIALECT_RULES) < prompts.SQL_SYSTEM.index(prompts.SCHEMA_RULES) < prompts.SQL_SYSTEM.index(prompts.OUTPUT_RULES)

# EXPLANATION_SYSTEM is a separate, unrelated prompt
assert "one short sentence" in prompts.EXPLANATION_SYSTEM
print("SQL_SYSTEM:", len(prompts.SQL_SYSTEM), "chars |", "EXPLANATION_SYSTEM:", len(prompts.EXPLANATION_SYSTEM), "chars")
print("OK")

SQL_SYSTEM: 1971 chars | EXPLANATION_SYSTEM: 460 chars
OK


## `format_examples`

In [3]:
# default (None) renders all FEW_SHOT_EXAMPLES
default_block = prompts.format_examples()
assert default_block.startswith("Examples of correct queries:")
for ex in prompts.FEW_SHOT_EXAMPLES:
    assert ex["question"] in default_block
    assert ex["sql"] in default_block

# empty list forces it off regardless of default examples
assert prompts.format_examples([]) == ""

# a custom example list is rendered instead of the default one
custom = [{"question": "Q?", "sql": "SELECT 1"}]
custom_block = prompts.format_examples(custom)
assert "Q?" in custom_block and "SELECT 1" in custom_block
assert prompts.FEW_SHOT_EXAMPLES[0]["question"] not in custom_block
print("OK")

OK


## No leakage from the eval test set

The module docstring warns that a few-shot example must never be copied from `eval/testset.yaml` -- that would be leakage, not generalisation. `eval/` doesn't exist in this repo yet, but this check is cheap to write now and will start actually verifying something the moment it does, rather than being a promise in a comment.

In [4]:
testset_path = Path.cwd().parent / "eval" / "testset.yaml"
if testset_path.exists():
    import yaml
    testset_questions = {row["question"] for row in yaml.safe_load(testset_path.read_text())}
    overlap = testset_questions & {ex["question"] for ex in prompts.FEW_SHOT_EXAMPLES}
    assert not overlap, f"few-shot examples leak from the eval test set: {overlap}"
    print("no overlap with eval/testset.yaml")
else:
    print("eval/testset.yaml does not exist yet -- nothing to check")

eval/testset.yaml does not exist yet -- nothing to check


## `build_sql_prompt`

In [5]:
schema_stub = "TABLE orders (order_id TEXT PK, order_status TEXT)"

# schema must come before the question (prompt-prefix caching relies on this)
p = prompts.build_sql_prompt("How many orders?", schema_stub, examples=[])
assert p.index("Database schema") < p.index("Question:")
assert p.rstrip().endswith("SQL:")
assert schema_stub in p

# examples=[] always forces examples off, regardless of settings.use_few_shot
original = settings.use_few_shot
try:
    settings.use_few_shot = True
    p_forced_off = prompts.build_sql_prompt("Q", schema_stub, examples=[])
    assert "Examples of correct queries" not in p_forced_off

    # examples=None follows settings.use_few_shot
    p_on = prompts.build_sql_prompt("Q", schema_stub)
    assert "Examples of correct queries" in p_on

    settings.use_few_shot = False
    p_off = prompts.build_sql_prompt("Q", schema_stub)
    assert "Examples of correct queries" not in p_off
finally:
    settings.use_few_shot = original

print("OK")

OK


## `build_correction_prompt`

In [6]:
# no history: just the failed query and its error
p = prompts.build_correction_prompt(
    "How many orders?", "SELECT * FORM orders", 'near "FORM": syntax error', schema_stub,
)
assert "Earlier attempts" not in p
assert "SELECT * FORM orders" in p
assert 'near "FORM": syntax error' in p

# history containing an entry equal to failed_sql is filtered out (it would
# be redundant with "Your most recent query" below it)
history = [
    {"sql": "SELECT * FORM orders", "error": "same as current, must be excluded"},
    {"sql": "SELECT cnt FROM orders", "error": "no such column: cnt"},
]
p2 = prompts.build_correction_prompt(
    "How many orders?", "SELECT * FORM orders", 'near "FORM": syntax error', schema_stub, history=history,
)
assert "Earlier attempts that also failed" in p2
assert "same as current, must be excluded" not in p2
assert "SELECT cnt FROM orders" in p2 and "no such column: cnt" in p2
print("OK")

OK


## `build_empty_result_prompt` and `render_rows_for_prompt`

In [7]:
p = prompts.build_empty_result_prompt("Revenue for 'eletronics'?", "SELECT * FROM orders WHERE 1=0", schema_stub)
assert "returned no rows" in p
assert "Decide which is true" in p

assert prompts.render_rows_for_prompt(None, []) == "(no columns)"
assert prompts.render_rows_for_prompt(["n"], []) == "(no rows)"

# NULL rendering and truncation past max_rows
rows = [[i, None] for i in range(25)]
rendered = prompts.render_rows_for_prompt(["id", "val"], rows, max_rows=20)
assert "NULL" in rendered
assert "and 5 more row(s)" in rendered
assert rendered.count("\n") == 1 + 20  # header + 20 body rows (the "more" suffix has no leading \n split)
print("OK")

OK


## `build_explanation_prompt`

In [8]:
sql_that_must_not_leak = "SELECT count(*) FROM orders WHERE order_status = 'delivered'"
p = prompts.build_explanation_prompt("How many delivered orders?", ["n"], [[96478]])
assert "96478" in p
assert sql_that_must_not_leak not in p  # the SQL itself should never be in this prompt
print("OK")

OK


## `prompt_metadata`

In [9]:
original = settings.use_few_shot
try:
    settings.use_few_shot = False
    meta_off = prompts.prompt_metadata()
    assert meta_off == {"prompt_version": prompts.PROMPT_VERSION, "use_few_shot": False, "few_shot_count": 0}

    settings.use_few_shot = True
    meta_on = prompts.prompt_metadata()
    assert meta_on["few_shot_count"] == len(prompts.FEW_SHOT_EXAMPLES)
finally:
    settings.use_few_shot = original
print("OK")

OK


## Integration: real schema -> `build_sql_prompt` -> `llm.complete` -> `execute_query`

This is the actual call path `nodes.py` will use once it exists. Makes one real API call.

In [10]:
import llm
from db import fetch_all, execute_query

def render_schema(tables: list[str]) -> str:
    lines = []
    for t in tables:
        cols = fetch_all(f"PRAGMA table_info({t})")
        col_strs = [f"{name} {coltype}" + (" PK" if pk else "") for _cid, name, coltype, _nn, _def, pk in cols]
        fks = fetch_all(f"PRAGMA foreign_key_list({t})")
        fk_strs = [f"{frm} -> {ref}({to})" for _id, _seq, ref, frm, to, *_ in fks]
        block = f"TABLE {t} (\n  " + ",\n  ".join(col_strs) + "\n)"
        if fk_strs:
            block += "\n  FOREIGN KEYS: " + "; ".join(fk_strs)
        lines.append(block)
    return "\n\n".join(lines)

schema_text = render_schema(["orders", "order_items", "products", "product_category_translation"])
question = "What is the total revenue for the English category 'toys'?"

user_prompt = prompts.build_sql_prompt(question, schema_text)
text = llm.complete(system=prompts.SQL_SYSTEM, user=user_prompt, max_tokens=300)
sql = llm.extract_sql(text)
print("generated sql:\n", sql)

result = execute_query(sql)
print("\nexecution result:", result.preview())
assert result.ok, f"generated SQL failed to execute: {result.error}"

generated sql:
 SELECT SUM(oi.price + oi.freight_value) AS total_revenue
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.product_id
JOIN product_category_translation AS pct ON p.product_category_name = pct.product_category_name
WHERE pct.product_category_name_english = 'toys'

execution result: 1 rows | 561372.55


In [11]:
# cross-check the result against a hand-written gold query
gold = execute_query(
    """
    SELECT round(sum(oi.price), 2)
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN product_category_translation t ON p.product_category_name = t.product_category_name
    WHERE t.product_category_name_english = 'toys'
    """
)
print("gold:", gold.preview())
print("model:", result.preview())
if result.rows and gold.rows:
    model_val = round(float(result.rows[0][0]), 2) if result.rows[0][0] is not None else None
    gold_val = gold.rows[0][0]
    print("match:" , model_val == gold_val)

gold: 1 rows | 483946.6
model: 1 rows | 561372.55
match: False


## More examples: the four patterns `prompts.py` targets

The module docstring for `FEW_SHOT_EXAMPLES` names four patterns that fail most often at baseline: the bridge-table join, the Portuguese/English category join, `strftime` date filtering, and distinct-customer counting. One example each, run through the real pipeline (`build_sql_prompt` -> `llm.complete` -> `execute_query`) and checked against a hand-written gold query.

In [12]:
examples = [
    {
        "pattern": "bridge table join",
        "question": "How many distinct sellers have shipped at least one item?",
        "gold": "SELECT COUNT(DISTINCT seller_id) FROM order_items",
        "tables": ["order_items"],
    },
    {
        "pattern": "Portuguese/English category join",
        "question": "How many products are in the English category 'bed_bath_table'?",
        "gold": (
            "SELECT COUNT(*) FROM products p "
            "JOIN product_category_translation t ON p.product_category_name = t.product_category_name "
            "WHERE t.product_category_name_english = 'bed_bath_table'"
        ),
        "tables": ["products", "product_category_translation"],
    },
    {
        "pattern": "strftime date filtering",
        "question": "How many orders were purchased in March 2018?",
        "gold": "SELECT COUNT(*) FROM orders WHERE strftime('%Y-%m', order_purchase_timestamp) = '2018-03'",
        "tables": ["orders"],
    },
    {
        "pattern": "distinct-customer counting",
        "question": "How many distinct customers (by customer_unique_id) are there in total?",
        "gold": "SELECT COUNT(DISTINCT customer_unique_id) FROM customers",
        "tables": ["customers"],
    },
]

rows = []
for ex in examples:
    schema = render_schema(ex["tables"])
    user_prompt = prompts.build_sql_prompt(ex["question"], schema)
    text = llm.complete(system=prompts.SQL_SYSTEM, user=user_prompt, max_tokens=300)
    sql = llm.extract_sql(text)

    result = execute_query(sql)
    gold_result = execute_query(ex["gold"])

    match = result.ok and gold_result.ok and result.rows == gold_result.rows
    rows.append({
        "pattern": ex["pattern"],
        "question": ex["question"],
        "generated_sql": sql,
        "model_result": result.preview(),
        "gold_result": gold_result.preview(),
        "match": match,
    })

import pandas as pd
pd.DataFrame(rows)[["pattern", "match", "model_result", "gold_result"]]

,pattern,match,model_result,gold_result
0,bridge table join,True,1 rows | 3095,1 rows | 3095
1,Portuguese/English category join,True,1 rows | 3029,1 rows | 3029
2,strftime date filtering,True,1 rows | 7211,1 rows | 7211
3,distinct-customer counting,True,1 rows | 96096,1 rows | 96096


## Integration: `build_correction_prompt` actually fixes a broken query

Feed the model a deliberately wrong query and the real `db.py` error, and confirm the correction prompt gets a working query back.

In [13]:
broken_sql = "SELECT COUNT(*) FROM orders WHERE order_stat = 'delivered'"
failed = execute_query(broken_sql)
print("broken query error:", failed.error)
assert not failed.ok

correction_prompt = prompts.build_correction_prompt(
    "How many delivered orders are there?", broken_sql, failed.error, schema_text,
)
text = llm.complete(system=prompts.SQL_SYSTEM, user=correction_prompt, max_tokens=300)
fixed_sql = llm.extract_sql(text)
print("corrected sql:", fixed_sql)

fixed_result = execute_query(fixed_sql)
print("corrected result:", fixed_result.preview())
assert fixed_result.ok, "the correction prompt should have produced runnable SQL"
assert fixed_sql != broken_sql, "the model must not resubmit the same broken query"

broken query error: no such column: order_stat. Did you mean: orders.order_id, orders.order_status, orders.order_approved_at?


corrected sql: SELECT COUNT(*) FROM orders WHERE order_status = 'delivered'
corrected result: 1 rows | 96478


In [14]:
print("total usage this notebook:", llm.usage_summary())

total usage this notebook: {'calls': 6, 'input_tokens': 3682, 'output_tokens': 193, 'cache_read_tokens': 0, 'cache_write_tokens': 0, 'cost_usd': 0.000668, 'cost_incomplete': False, 'mean_latency_ms': 1416.8, 'model': 'gpt-4o-mini', 'pricing_verified_on': '2026-08-05'}
